# B3/B3L: QLoRA seeds 42/43/44, then generation

Prepare `sft-train.jsonl`, `sft-dev.jsonl` and `prompts-dev_eval.jsonl` for both configs locally with `eth-nl2sql prompts --config b3 --training` (repeat for b3l) and `eth-nl2sql prompts --config b3 --split dev_eval` (repeat for b3l).

Requires a T4 GPU, internet, a Kaggle secret `HF_TOKEN` with model access, and the repo attached as `eth-nl2sql-src`, including `runs/<config>/sft-*.jsonl` and `prompts-dev_eval.jsonl`. Test v2 remains gated while draft. Preflight downloads the pinned tokenizer but does not load a model.
Each seed writes to `runs/<config>/seed-<seed>/`. Keep the manifests, common SFT inputs and adapters together when copying results back. `--resume` validates existing runs and starts seeds that have no artifacts yet.

In [ ]:
!cp -r /kaggle/input/eth-nl2sql-src /kaggle/working/eth-nl2sql
%cd /kaggle/working/eth-nl2sql
!pip install -q -e '.[gpu]' 

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(UserSecretsClient().get_secret('HF_TOKEN'))

In [ ]:
import subprocess

CONFIGS = ['b3', 'b3l']
for name in CONFIGS:
    subprocess.run(['eth-nl2sql', 'preflight', '--config', name, '--training', '--seed', 'all'], check=True)
subprocess.run(['eth-nl2sql', 'train', '--config', 'all', '--seed', 'all', '--resume'], check=True)

In [ ]:
for name in CONFIGS:
    # Automatically loads the matching adapter for each training seed.
    subprocess.run(['eth-nl2sql', 'generate', '--config', name, '--split', 'dev_eval', '--seed', 'all'], check=True)

In [ ]:
import shutil
# Includes checkpoints, adapters, inputs and manifests for verified resume/evaluation locally.
shutil.make_archive('/kaggle/working/qlora-three-seeds', 'zip', '.', 'runs')